In [1]:
#!/usr/bin/env python
# coding: utf-8

"""
Multi-Coverage Position-Wise Accuracy Analysis
==============================================

Goal: Verify the interpretation by testing at multiple coverage levels (M=10, 15, 20, 25)

If alignment interpretation is correct:
- Edge > Center pattern should PERSIST across all coverage levels
- The gap might DECREASE with higher coverage (more averaging)

If Bi-LSTM exploits context:
- Bi-LSTM's relative advantage in center should persist across coverage levels
"""

import os
import random
import pickle
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import json

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# =============================================================================
# DEVICE CONFIGURATION
# =============================================================================
DEVICE_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = DEVICE_ID

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

# =============================================================================
# CONFIGURATION
# =============================================================================
CONFIG = {
    "error_name": "EZ17",
    "alphabet_mode": "2mix_3mix_4mix",
    "vocab_size": 15,
    "seq_length": 136,
    
    # Multiple coverage levels to test
    "coverage_levels": [10, 15, 20, 25],
    
    # Model Architecture
    "input_channels": 4,
    "hidden_dim": 128,
    "num_layers": 2,
    "dropout": 0.2,
    "bidirectional": True,
    
    # Dataset
    "num_samples": 100000,
    "max_coverage": 25,
    
    "batch_size": 500,
    "edge_size": 15,
    
    "dataset_dir": "./dataset",
    "results_dir": "./results_EZ17_2mix_3mix_4mix",
    "output_dir": "./position_analysis_multi_coverage",
    
    "seed": 42
}

CONFIG["dataset_path"] = (f"{CONFIG['dataset_dir']}/"
                          f"dna_{CONFIG['error_name']}_{CONFIG['alphabet_mode']}_"
                          f"{CONFIG['num_samples']}_{CONFIG['max_coverage']}.pkl")

os.makedirs(CONFIG['output_dir'], exist_ok=True)

# =============================================================================
# SEED
# =============================================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG['seed'])

# =============================================================================
# SYMBOL MAPPINGS & IDEAL VECTORS
# =============================================================================
def build_symbol_to_idx():
    symbol_to_idx = {
        'A': 0, 'C': 1, 'G': 2, 'T': 3,
        'M1': 4, 'M2': 5, 'M3': 6, 'M4': 7, 'M5': 8, 'M6': 9,
        'T1': 10, 'T2': 11, 'T3': 12, 'T4': 13,
        'Q1': 14
    }
    return symbol_to_idx

def build_ideal_vectors():
    third = 1.0 / 3.0
    ideal_vectors = [
        [1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0],
        [0.5, 0.0, 0.0, 0.5], [0.0, 0.5, 0.5, 0.0],
        [0.0, 0.5, 0.0, 0.5], [0.0, 0.0, 0.5, 0.5],
        [0.5, 0.5, 0.0, 0.0], [0.5, 0.0, 0.5, 0.0],
        [third, third, third, 0.0], [third, third, 0.0, third],
        [third, 0.0, third, third], [0.0, third, third, third],
        [0.25, 0.25, 0.25, 0.25],
    ]
    return torch.tensor(ideal_vectors, dtype=torch.float32)

SYMBOL_TO_IDX = build_symbol_to_idx()
IDEAL_VECTORS = build_ideal_vectors().to(device)

# =============================================================================
# DATA PREPROCESSING
# =============================================================================
def preprocess_cluster_to_matrix(cluster_reads, target_length):
    profile_matrix = np.zeros((4, target_length), dtype=np.float32)
    base_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    num_reads = len(cluster_reads)
    
    for read in cluster_reads:
        read_len = len(read)
        if read_len == 0:
            continue
        for t_idx in range(target_length):
            read_idx = int((t_idx + 0.5) * (read_len / target_length))
            if read_idx >= read_len:
                read_idx = read_len - 1
            base = read[read_idx]
            if base in base_map:
                profile_matrix[base_map[base], t_idx] += 1.0
    
    if num_reads > 0:
        profile_matrix /= num_reads
    return profile_matrix

# =============================================================================
# DATASET
# =============================================================================
class CompositeDNADataset(Dataset):
    def __init__(self, data_path, seq_length, symbol_to_idx, limit_coverage=None):
        with open(data_path, 'rb') as f:
            raw_data = pickle.load(f)
        self.samples = raw_data['data']
        self.seq_length = seq_length
        self.symbol_to_idx = symbol_to_idx
        self.limit_coverage = limit_coverage
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        item = self.samples[idx]
        cluster = item['cluster']
        if self.limit_coverage is not None:
            cluster = cluster[:min(self.limit_coverage, len(cluster))]
        x_data = preprocess_cluster_to_matrix(cluster, self.seq_length)
        label_seq = item['label']
        y_data = np.array([self.symbol_to_idx[s] for s in label_seq], dtype=np.longlong)
        return torch.tensor(x_data, dtype=torch.float32), torch.tensor(y_data, dtype=torch.long)

# =============================================================================
# MODEL
# =============================================================================
class CompositeDecoderLSTM(nn.Module):
    def __init__(self, config):
        super(CompositeDecoderLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=config['input_channels'],
            hidden_size=config['hidden_dim'],
            num_layers=config['num_layers'],
            batch_first=True,
            bidirectional=config['bidirectional'],
            dropout=config['dropout'] if config['num_layers'] > 1 else 0
        )
        fc_in = config['hidden_dim'] * 2 if config['bidirectional'] else config['hidden_dim']
        self.fc = nn.Linear(fc_in, config['vocab_size'])
        
    def forward(self, x):
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        logits = self.fc(out)
        return logits.permute(0, 2, 1)

# =============================================================================
# BASELINE DECODERS
# =============================================================================
def min_distance_decoder(obs, ideal_vectors):
    dists = torch.sum((obs.unsqueeze(2) - ideal_vectors.unsqueeze(0).unsqueeze(0)) ** 2, dim=3)
    return torch.argmin(dists, dim=2)

def kl_divergence_decoder(obs, ideal_vectors, epsilon=0.01):
    ideal_safe = torch.clamp(ideal_vectors.clone(), min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    cross_entropy = -(obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmin(cross_entropy, dim=-1)

# =============================================================================
# POSITION-WISE ACCURACY
# =============================================================================
def compute_position_wise_accuracy(model, loader, ideal_vectors, device, seq_length):
    model.eval()
    
    correct_lstm = np.zeros(seq_length)
    correct_mindist = np.zeros(seq_length)
    correct_kl = np.zeros(seq_length)
    total_per_pos = np.zeros(seq_length)
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            batch_size = inputs.size(0)
            obs = inputs.permute(0, 2, 1)
            
            outputs = model(inputs)
            pred_lstm = torch.argmax(outputs, dim=1)
            pred_mindist = min_distance_decoder(obs, ideal_vectors)
            pred_kl = kl_divergence_decoder(obs, ideal_vectors)
            
            labels_np = labels.cpu().numpy()
            pred_lstm_np = pred_lstm.cpu().numpy()
            pred_mindist_np = pred_mindist.cpu().numpy()
            pred_kl_np = pred_kl.cpu().numpy()
            
            for pos in range(seq_length):
                correct_lstm[pos] += np.sum(pred_lstm_np[:, pos] == labels_np[:, pos])
                correct_mindist[pos] += np.sum(pred_mindist_np[:, pos] == labels_np[:, pos])
                correct_kl[pos] += np.sum(pred_kl_np[:, pos] == labels_np[:, pos])
                total_per_pos[pos] += batch_size
    
    return {
        'lstm': 100 * correct_lstm / total_per_pos,
        'mindist': 100 * correct_mindist / total_per_pos,
        'kl': 100 * correct_kl / total_per_pos,
        'positions': np.arange(seq_length)
    }

def analyze_regions(position_accuracies, edge_size, seq_length):
    edge_left = list(range(0, edge_size))
    edge_right = list(range(seq_length - edge_size, seq_length))
    edge_positions = edge_left + edge_right
    center_positions = list(range(edge_size, seq_length - edge_size))
    
    results = {}
    for decoder in ['lstm', 'mindist', 'kl']:
        acc = position_accuracies[decoder]
        edge_acc = np.mean(acc[edge_positions])
        center_acc = np.mean(acc[center_positions])
        overall_acc = np.mean(acc)
        
        results[decoder] = {
            'edge': edge_acc,
            'center': center_acc,
            'overall': overall_acc,
            'center_minus_edge': center_acc - edge_acc
        }
    
    return results

# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    
    print("\n" + "="*70)
    print("🔬 MULTI-COVERAGE POSITION-WISE ANALYSIS")
    print("="*70)
    
    # Check dataset exists
    if not os.path.exists(CONFIG['dataset_path']):
        raise FileNotFoundError(f"❌ Dataset not found: {CONFIG['dataset_path']}")
    
    # Store results for all coverage levels
    all_results = {}
    
    for coverage in CONFIG['coverage_levels']:
        model_path = (f"{CONFIG['results_dir']}/"
                      f"best_model_{CONFIG['error_name']}_{CONFIG['alphabet_mode']}_M{coverage}.pth")
        
        if not os.path.exists(model_path):
            print(f"⚠️  Model not found for M={coverage}: {model_path}")
            continue
        
        print(f"\n{'='*60}")
        print(f"📊 ANALYZING COVERAGE M = {coverage}")
        print(f"{'='*60}")
        
        # Load data
        set_seed(CONFIG['seed'])
        full_ds = CompositeDNADataset(
            CONFIG['dataset_path'],
            CONFIG['seq_length'],
            SYMBOL_TO_IDX,
            limit_coverage=coverage
        )
        
        train_size = int(0.8 * len(full_ds))
        val_size = len(full_ds) - train_size
        _, val_ds = random_split(full_ds, [train_size, val_size])
        val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False)
        
        # Load model
        model = CompositeDecoderLSTM(CONFIG).to(device)
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval()
        
        # Compute position-wise accuracy
        position_accuracies = compute_position_wise_accuracy(
            model, val_loader, IDEAL_VECTORS, device, CONFIG['seq_length']
        )
        
        # Analyze regions
        region_results = analyze_regions(
            position_accuracies, CONFIG['edge_size'], CONFIG['seq_length']
        )
        
        all_results[coverage] = {
            'position_accuracies': {k: v.tolist() if isinstance(v, np.ndarray) else v 
                                   for k, v in position_accuracies.items()},
            'region_results': region_results
        }
        
        # Print summary for this coverage
        print(f"\n   {'Decoder':<15} {'Edge':<10} {'Center':<10} {'Overall':<10} {'Δ(C-E)':<10}")
        print(f"   {'-'*55}")
        for decoder in ['lstm', 'kl', 'mindist']:
            r = region_results[decoder]
            name = {'lstm': 'Bi-LSTM', 'kl': 'KL/ML', 'mindist': 'Min. Distance'}[decoder]
            print(f"   {name:<15} {r['edge']:<10.2f} {r['center']:<10.2f} {r['overall']:<10.2f} {r['center_minus_edge']:+.2f}")
    
    # =============================================================================
    # SUMMARY COMPARISON
    # =============================================================================
    print(f"\n\n{'='*80}")
    print(f"📊 SUMMARY: EDGE VS CENTER ACROSS COVERAGE LEVELS")
    print(f"{'='*80}")
    
    print(f"\n{'Coverage':<10} | {'Bi-LSTM':<25} | {'KL/ML':<25} | {'Min. Distance':<25}")
    print(f"{'':<10} | {'Edge':>8} {'Center':>8} {'Δ':>7} | {'Edge':>8} {'Center':>8} {'Δ':>7} | {'Edge':>8} {'Center':>8} {'Δ':>7}")
    print(f"{'-'*95}")
    
    for coverage in CONFIG['coverage_levels']:
        if coverage not in all_results:
            continue
        r = all_results[coverage]['region_results']
        lstm = r['lstm']
        kl = r['kl']
        mindist = r['mindist']
        print(f"M={coverage:<7} | {lstm['edge']:>8.2f} {lstm['center']:>8.2f} {lstm['center_minus_edge']:>+7.2f} | "
              f"{kl['edge']:>8.2f} {kl['center']:>8.2f} {kl['center_minus_edge']:>+7.2f} | "
              f"{mindist['edge']:>8.2f} {mindist['center']:>8.2f} {mindist['center_minus_edge']:>+7.2f}")
    
    # =============================================================================
    # KEY INSIGHTS
    # =============================================================================
    print(f"\n\n{'='*80}")
    print(f"🔍 KEY INSIGHTS")
    print(f"{'='*80}")
    
    # Check if Edge > Center pattern persists
    edge_gt_center_count = 0
    total_count = 0
    for coverage in CONFIG['coverage_levels']:
        if coverage not in all_results:
            continue
        for decoder in ['lstm', 'kl', 'mindist']:
            r = all_results[coverage]['region_results'][decoder]
            if r['edge'] > r['center']:
                edge_gt_center_count += 1
            total_count += 1
    
    print(f"\n1. Edge > Center Pattern:")
    print(f"   Occurs in {edge_gt_center_count}/{total_count} cases ({100*edge_gt_center_count/total_count:.0f}%)")
    
    # Check Bi-LSTM advantage in center
    print(f"\n2. Bi-LSTM Advantage (over KL/ML) by Region:")
    print(f"   {'Coverage':<10} {'Edge Adv.':<15} {'Center Adv.':<15} {'Center > Edge?':<15}")
    print(f"   {'-'*55}")
    
    center_advantage_higher = 0
    for coverage in CONFIG['coverage_levels']:
        if coverage not in all_results:
            continue
        r = all_results[coverage]['region_results']
        edge_adv = r['lstm']['edge'] - r['kl']['edge']
        center_adv = r['lstm']['center'] - r['kl']['center']
        higher = "✓ YES" if center_adv > edge_adv else "✗ NO"
        if center_adv > edge_adv:
            center_advantage_higher += 1
        print(f"   M={coverage:<7} {edge_adv:>+10.2f}%    {center_adv:>+10.2f}%    {higher}")
    
    print(f"\n   Bi-LSTM advantage larger in center: {center_advantage_higher}/{len([c for c in CONFIG['coverage_levels'] if c in all_results])} coverage levels")
    
    # =============================================================================
    # CONCLUSION
    # =============================================================================
    print(f"\n\n{'='*80}")
    print(f"📝 CONCLUSION")
    print(f"{'='*80}")
    
    if edge_gt_center_count == total_count:
        print(f"\n✓ Edge > Center pattern is UNIVERSAL (all decoders, all coverage levels)")
        print(f"  → This is due to ALIGNMENT/PREPROCESSING, not decoder-specific")
    
    if center_advantage_higher >= len([c for c in CONFIG['coverage_levels'] if c in all_results]) / 2:
        print(f"\n✓ Bi-LSTM advantage is LARGER in CENTER than at EDGES")
        print(f"  → This proves Bi-LSTM uses SEQUENTIAL CONTEXT to compensate for")
        print(f"     alignment-induced noise in the more challenging center positions")
    
    # Save results
    results_path = os.path.join(CONFIG['output_dir'], 'multi_coverage_results.json')
    with open(results_path, 'w') as f:
        json.dump(all_results, f, indent=4)
    print(f"\n💾 Results saved: {results_path}")
    
    # =============================================================================
    # PLOT: Edge vs Center Gap by Coverage
    # =============================================================================
    plt.figure(figsize=(12, 6))
    
    coverages = [c for c in CONFIG['coverage_levels'] if c in all_results]
    lstm_gaps = [all_results[c]['region_results']['lstm']['center_minus_edge'] for c in coverages]
    kl_gaps = [all_results[c]['region_results']['kl']['center_minus_edge'] for c in coverages]
    mindist_gaps = [all_results[c]['region_results']['mindist']['center_minus_edge'] for c in coverages]
    
    x = np.arange(len(coverages))
    width = 0.25
    
    plt.bar(x - width, lstm_gaps, width, label='Bi-LSTM', color='#2ecc71')
    plt.bar(x, kl_gaps, width, label='KL/ML', color='#3498db')
    plt.bar(x + width, mindist_gaps, width, label='Min. Distance', color='#e74c3c')
    
    plt.axhline(y=0, color='black', linestyle='-', linewidth=1)
    plt.xlabel('Coverage (M)', fontsize=12)
    plt.ylabel('Center − Edge Accuracy (%)', fontsize=12)
    plt.title('Position Dependence (Center − Edge Gap) Across Coverage Levels\nNegative = Edge Better Than Center', fontsize=14)
    plt.xticks(x, [f'M={c}' for c in coverages])
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plot_path = os.path.join(CONFIG['output_dir'], 'edge_center_gap_by_coverage.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📈 Plot saved: {plot_path}")
    
    # =============================================================================
    # PLOT: Bi-LSTM Advantage by Region and Coverage
    # =============================================================================
    plt.figure(figsize=(10, 6))
    
    edge_advs = [all_results[c]['region_results']['lstm']['edge'] - all_results[c]['region_results']['kl']['edge'] for c in coverages]
    center_advs = [all_results[c]['region_results']['lstm']['center'] - all_results[c]['region_results']['kl']['center'] for c in coverages]
    
    plt.plot(coverages, edge_advs, 'o--', lw=2, ms=8, color='#e74c3c', label='Edge Region Advantage')
    plt.plot(coverages, center_advs, 's-', lw=2, ms=8, color='#2ecc71', label='Center Region Advantage')
    
    plt.xlabel('Coverage (M)', fontsize=12)
    plt.ylabel('Bi-LSTM Advantage over KL/ML (%)', fontsize=12)
    plt.title('Bi-LSTM Advantage Over KL/ML by Region\n(Higher in Center = Context Exploitation)', fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.xticks(coverages)
    
    plt.tight_layout()
    plot_path = os.path.join(CONFIG['output_dir'], 'bilstm_advantage_by_region.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📈 Plot saved: {plot_path}")
    
    print(f"\n✅ Analysis complete!")

✅ Using device: cuda

🔬 MULTI-COVERAGE POSITION-WISE ANALYSIS

📊 ANALYZING COVERAGE M = 10

   Decoder         Edge       Center     Overall    Δ(C-E)    
   -------------------------------------------------------
   Bi-LSTM         95.21      91.70      92.48      -3.51
   KL/ML           88.64      83.57      84.69      -5.07
   Min. Distance   82.12      80.27      80.68      -1.85

📊 ANALYZING COVERAGE M = 15

   Decoder         Edge       Center     Overall    Δ(C-E)    
   -------------------------------------------------------
   Bi-LSTM         98.19      96.37      96.77      -1.83
   KL/ML           95.87      91.84      92.73      -4.02
   Min. Distance   90.08      88.46      88.82      -1.63

📊 ANALYZING COVERAGE M = 20

   Decoder         Edge       Center     Overall    Δ(C-E)    
   -------------------------------------------------------
   Bi-LSTM         99.19      98.23      98.44      -0.95
   KL/ML           97.87      94.30      95.09      -3.57
   Min. Distance  